# YOLO26s CUDA training
Run setup and dataset verification first. Then explicitly choose **one** action (`smoke`, `development`, or `resume`) in the configuration cell and run its corresponding section. The default `none` makes Run All safe: no training is launched. Results inspection is independent. Never use the held-out test split for training or model selection.

In [ ]:
import csv
import importlib.util
import json
import platform
import subprocess
from pathlib import Path

import matplotlib.pyplot as plt
import torch
import ultralytics
import yaml
from IPython.display import Image as DisplayImage, display
from ultralytics import YOLO
from ultralytics.cfg import get_cfg
from grid_detection.dataset.yolo import YOLO_CLASSES
from grid_detection.research import DATA_DIR, PROJECT_ROOT, RUNS_DIR

print(f"Python {platform.python_version()} | PyTorch {torch.__version__} | Ultralytics {ultralytics.__version__}")
print(f"CUDA runtime: {torch.version.cuda}; available: {torch.cuda.is_available()}")
for i in range(torch.cuda.device_count()):
    gpu = torch.cuda.get_device_properties(i)
    print(f"GPU {i}: {gpu.name}; {gpu.total_memory / 2**30:.2f} GiB")

## Editable configuration
Set `ACTION` only when ready. Smoke and development runs use separate directories and pretrained initializations. Change run names to start additional runs; existing paths are refused. The first model load may download the pretrained weights.

In [ ]:
ACTION = "none"  # "none", "smoke", "development", or "resume"
if ACTION not in {"none", "smoke", "development", "resume"}:
    raise ValueError(f"Unknown ACTION: {ACTION}")
model_name = "yolo26s.pt"
smoke_run_name = "yolo26s_smoke_v4"
development_run_name = "yolo26s_development_v1"
resume_run_name = development_run_name  # existing interrupted run only
resume_dataset = "development"  # "smoke" or "development", matching the interrupted run
inspect_run_name = development_run_name

datasets = {
    "smoke": ("smoke_v4", {"train": 20, "val": 5, "test": 5}),
    "development": ("development_v1", {"train": 2000, "val": 500, "test": 500}),
}
for run_name in (smoke_run_name, development_run_name, resume_run_name, inspect_run_name):
    if not run_name or run_name in {".", ".."} or Path(run_name).name != run_name:
        raise ValueError(f"Use a single run directory name: {run_name!r}")

training = dict(imgsz=1280, batch=8, epochs=100, patience=20, device=0,
                workers=4, amp=True, seed=42, deterministic=True, optimizer="auto",
                hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, translate=0.05, scale=0.1,
                degrees=0.0, shear=0.0, perspective=0.0, flipud=0.0, fliplr=0.0,
                bgr=0.0, mosaic=0.0, mixup=0.0, cutmix=0.0, copy_paste=0.0,
                plots=True, val=True, save=True)
get_cfg(overrides=training)  # Validate argument names against installed Ultralytics.
print(f"Action: {ACTION}; runs: {RUNS_DIR}")

The detection policy retains HSV, small translation and scale changes. Rotation, shear, perspective, flips, channel swaps, mosaic and image mixing are disabled. `auto_augment` and `erasing` in this Ultralytics version apply to classification, not this detection task. The generator's capture augmentation is unchanged. The locked environment does not include Albumentations; if it is installed separately, Ultralytics would apply extra blur, grayscale and CLAHE transforms. Training cells reject that environment to keep the policy reproducible.

## Dataset verification
Checks the chosen dataset before a new run. For inspection with `ACTION = "none"`, set `dataset_to_check`. `data.yaml` contains an absolute path written by the generator, so generate the dataset on WSL and verify that it still points to this local dataset.

In [ ]:
dataset_to_check = resume_dataset if ACTION == "resume" else ("development" if ACTION == "none" else ACTION)
name, counts = datasets[dataset_to_check]
dataset_root = DATA_DIR / name
config_path = dataset_root / "config.json"
if ACTION != "none" or config_path.exists():
    dataset_config = json.loads(config_path.read_text())
    expected = dict(generator_revision=4, status="complete", seed=42,
                    png_compression=1, scene_details=True, augment=True,
                    max_attempts=20, **counts)
    for key, value in expected.items():
        if dataset_config.get(key) != value:
            raise ValueError(f"{config_path}: {key} expected {value!r}, got {dataset_config.get(key)!r}")
    if dataset_config.get("classes") != list(YOLO_CLASSES) or tuple(YOLO_CLASSES) != (
        "game_region", "grid", "mine_counter", "reset_button", "timer"):
        raise ValueError("Dataset class order differs from the experiment")
    data_yaml = dataset_root / "data.yaml"
    data = yaml.safe_load(data_yaml.read_text())
    if data.get("names") != dict(enumerate(YOLO_CLASSES)):
        raise ValueError("Unexpected data.yaml class order")
    if Path(data["path"]).resolve() != dataset_root.resolve():
        raise ValueError("data.yaml path does not point to this local dataset")
    for split in counts:
        if data.get(split) != f"images/{split}":
            raise ValueError(f"Unexpected {split} path in data.yaml")
        for folder, extension in (("images", "png"), ("labels", "txt")):
            if len(list((dataset_root / folder / split).glob(f"*.{extension}"))) != counts[split]:
                raise ValueError(f"Unexpected {split} {folder} count")
    for required in ("manifest.jsonl", "statistics.json"):
        if not (dataset_root / required).is_file():
            raise FileNotFoundError(dataset_root / required)
    print(f"Verified {name}: {counts}; workers used for generation: {dataset_config['workers']}")
else:
    print(f"No dataset at {dataset_root}; generate it before training.")

## Smoke training — one epoch
Set `ACTION = "smoke"` and run setup, verification and this cell. It uses the same image size, batch, workers, CUDA and augmentation policy as development. Check `weights/`, `args.yaml`, `results.csv` and plots after completion.

In [ ]:
if ACTION == "smoke":
    if importlib.util.find_spec("albumentations") is not None:
        raise RuntimeError("Remove optional Albumentations to preserve the documented augmentation policy")
    if not torch.cuda.is_available():
        raise RuntimeError("Training requires a working CUDA device; CPU/MPS fallback is disabled")
    run_dir = RUNS_DIR / smoke_run_name
    run_dir.mkdir(parents=True, exist_ok=False)
    options = dict(training, epochs=1, data=str(data_yaml), project=str(RUNS_DIR),
                   name=smoke_run_name, exist_ok=True)
    model = YOLO(model_name)
    model.train(**options)
    print(f"Smoke artifacts: {run_dir}")

## Development training
Set `ACTION = "development"` and run setup, verification and this cell. The new model starts from pretrained weights, never from the smoke checkpoint. The directory is reserved before loading weights; a failed launch leaves it in place to prevent accidental reuse.

In [ ]:
if ACTION == "development":
    if importlib.util.find_spec("albumentations") is not None:
        raise RuntimeError("Remove optional Albumentations to preserve the documented augmentation policy")
    if not torch.cuda.is_available():
        raise RuntimeError("Training requires a working CUDA device; CPU/MPS fallback is disabled")
    commit = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=PROJECT_ROOT, text=True).strip()
    dirty = bool(subprocess.check_output(["git", "status", "--porcelain"], cwd=PROJECT_ROOT, text=True).strip())
    run_dir = RUNS_DIR / development_run_name
    run_dir.mkdir(parents=True, exist_ok=False)
    options = dict(training, data=str(data_yaml), project=str(RUNS_DIR),
                   name=development_run_name, exist_ok=True)
    metadata = dict(dataset_name=name, dataset_config=dataset_config, generator_revision=4,
                    dataset_seed=dataset_config["seed"], git_commit=commit, working_tree_dirty=dirty,
                    model_initialization=model_name, python=platform.python_version(),
                    pytorch=torch.__version__, ultralytics=ultralytics.__version__,
                    cuda_runtime=torch.version.cuda, training_configuration=options)
    (run_dir / "experiment.json").write_text(json.dumps(metadata, indent=2) + "\n")
    model = YOLO(model_name)
    model.train(**options)
    print(f"Development artifacts: {run_dir}")

## Explicitly resume an interrupted run
Set `ACTION = "resume"` and run setup, verification and this cell only when resuming the original run. Native resume restores training state and saved arguments; it is not a way to change the experiment configuration. Completed runs cannot normally be resumed. To initialize a new run from weights instead, use a new run name and explicitly set `model_name` to those weights; that is a new training run, not resume. Keep the original dataset at its recorded path.

In [ ]:
if ACTION == "resume":
    if importlib.util.find_spec("albumentations") is not None:
        raise RuntimeError("Remove optional Albumentations to preserve the documented augmentation policy")
    if not torch.cuda.is_available():
        raise RuntimeError("Resume requires a working CUDA device")
    checkpoint = RUNS_DIR / resume_run_name / "weights" / "last.pt"
    if not checkpoint.is_file():
        raise FileNotFoundError(checkpoint)
    model = YOLO(str(checkpoint))
    if model.ckpt.get("epoch", -1) < 0 or model.ckpt.get("optimizer") is None:
        raise ValueError("Checkpoint lacks resumable epoch/optimizer state; it may be a completed run")
    saved_data = model.ckpt.get("train_args", {}).get("data")
    if not saved_data or Path(saved_data).resolve() != data_yaml.resolve():
        raise ValueError("Checkpoint dataset does not match resume_dataset; select the original dataset")
    model.train(resume=True)

## Inspect training artifacts
Set `inspect_run_name` above and run this section after training. Curves come from the actual CSV; the validation results used during training are not held-out test results. Ultralytics selects `best.pt` by validation fitness (in this installed detection version, mAP50–95). Exported final checkpoints may have their epoch stripped, so do not infer the exact best epoch from a rounded CSV metric.

In [ ]:
inspect_dir = RUNS_DIR / inspect_run_name
csv_path = inspect_dir / "results.csv"
if csv_path.is_file():
    with csv_path.open(newline="") as f:
        rows = list(csv.DictReader(f, skipinitialspace=True))
    if rows:
        rows = [{k.strip(): v for k, v in row.items()} for row in rows]
        epochs = [float(row["epoch"]) for row in rows]
        groups = {
            "Loss": [k for k in rows[0] if k.startswith(("train/", "val/")) and "loss" in k],
            "Precision and recall": [k for k in rows[0] if k.startswith(("metrics/precision", "metrics/recall"))],
            "mAP": [k for k in rows[0] if k.startswith(("metrics/mAP50(", "metrics/mAP50-95("))],
            "Learning rate": [k for k in rows[0] if k.startswith("lr/")],
        }
        fig, axes = plt.subplots(2, 2, figsize=(12, 8))
        for ax, (title, keys) in zip(axes.flat, groups.items()):
            for key in keys:
                ax.plot(epochs, [float(row[key]) for row in rows], label=key)
            ax.set(title=title, xlabel="Epoch")
            if keys: ax.legend(fontsize=8)
        fig.tight_layout()
        plt.show()
        if "time" in rows[-1]:
            print(f"Reported elapsed training time: {float(rows[-1]['time']):.1f} s")
    print(f"Best checkpoint: {inspect_dir / 'weights' / 'best.pt'}")
    print(f"Last checkpoint: {inspect_dir / 'weights' / 'last.pt'}")
    print("Exact best epoch: not reliably available from standard stripped artifacts; inspect the original checkpoint metadata if retained.")
    for plot in ("results.png", "confusion_matrix.png", "confusion_matrix_normalized.png", "BoxPR_curve.png", "BoxP_curve.png", "BoxR_curve.png"):
        path = inspect_dir / plot
        if path.is_file(): display(DisplayImage(filename=str(path)))
else:
    print(f"No training results at {csv_path}")